<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Document_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Document Loader

 in this note book i am going to implement pipline for injection and preprocessing of `pdf` using `PyMupdf `

***Script that ingests a PDF, extracts text, splits into question-level chunks, and saves as structured JSON — zero libraries except PyMuPDF.***

 ### setup

In [ ]:
%%capture
!pip install pymupdf

In [ ]:
import pymupdf
print(pymupdf.__doc__)

In [ ]:
%%capture
!pip install -U langchain-text-splitters
!pip install langchain-community

## scripts for opening and preprocessing the pdf document

In [ ]:
# loads the document
doc = pymupdf.open("/content/drive/MyDrive/LLm training datasets /Copy of AI Engineering.pdf")

Extracting text from Pdf

In [ ]:
doc = pymupdf.open("/content/drive/MyDrive/LLm training datasets /Copy of AI Engineering.pdf")
# cretating a text output
out = open("extracted_text.txt","wb")
# iterating over pages
for page in doc:
  tp = page.get_textpage_ocr()
  text = page.get_text(textpage = tp).encode("utf8") # get plain text (is in UTF-8)
  out.write(text) # write text of pages
  out.write(f"\n--- Page {page.number + 1} ---\n".encode("utf8"))

out.close()

In [ ]:
with open("extracted_text.txt","r") as f:
    full_document_content = f.read()
# The `extracted_doc` variable (file object) is no longer needed after reading its content into `full_document_content`.

## Chunking the text

In [ ]:
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 512, chunk_overlap = 100)
chunks = text_splitter.split_text(full_document_content)


print(f"Total chunks: {len(chunks)}")
print(chunks[0])



## Adding Metadata

In [ ]:
from datetime import date
from langchain_core.documents import Document

# ── fill these for your document ──────────────────────────────────────
SOURCE_PATH  = "Ai engineering.pdf"
DOC_TYPE     = "book"          # e.g. "report", "invoice", "manual"
DOC_TITLE    = "Ai"
DOC_AUTHOR   = "Harsh Prajapati"
CREATION_DATE = "2005-12-20"       # ISO format from pdf.metadata["creationDate"]
TOTAL_PAGES  = 1                  # from pymupdf doc.page_count

total  = len(chunks)
today  = date.today().isoformat()

# wrap each plain string → Document with full metadata
docs = []
for idx, chunk_text in enumerate(chunks):
    meta = {
        # document-level (same for every chunk)
        "source":         SOURCE_PATH,
        "title":          DOC_TITLE,
        "author":         DOC_AUTHOR,
        "creation_date":  CREATION_DATE,
        "total_pages":    TOTAL_PAGES,
        "doc_type":       DOC_TYPE,

        # chunk-level (unique per chunk)
        "chunk_index":    idx,
        "total_chunks":   total,
        "char_count":     len(chunk_text),
        "prev_chunk_id":  idx - 1 if idx > 0          else None,
        "next_chunk_id":  idx + 1 if idx < total - 1  else None,
        "ingestion_date": today,
    }
    docs.append(Document(page_content=chunk_text, metadata=meta))

# verify
print(f"Total documents: {len(docs)}")
print(f"\nSample metadata (chunk 271):")
for k, v in docs[1].metadata.items():
    print(f"  {k:<18} {v}")
print(f"\nSample content (chunk 2:\n{docs[1].page_content}")

our dataloader is ready !!

print 20 random chunks

In [ ]:
for doc in docs[:5]:
    print("----")
    print(doc.page_content)
    print(doc.metadata)